# Fine-tuning runs on Colab (A100)

Runs `scripts/run_finetune.py` for the configs below on the branch pushed to GitHub, exactly as locally.
`PARALLEL` runs share the GPU (one process each; a bag run of ResNet-18 needs ~2 GB, ConvNeXt-T ~5 GB of the 40 GB).
Results (`outputs/<name>/`, `reports/results.csv`, `logs/`) and the tile cache are copied to Google Drive whenever a run
finishes, so a disconnected session loses only the runs in progress. A progress line (runs, folds, ETA) is printed every minute; each run's own output is in `logs/<name>.log`
(Files panel → repo/logs, or `tail -f /content/repo/logs/*.log` in the Colab terminal).

Before the first run: **Runtime → Change runtime type → A100 GPU**, and add two Colab secrets (🔑 panel on the left):
`KAGGLE_API_TOKEN` (a `KGAT_...` token from kaggle.com → Settings → API Tokens), with notebook access on.
Hypotheses for the block must be committed **before** these runs (HYPOTHESES.md).

In [ ]:
BRANCH = "part-1"
CONFIGS = [  # block L: leave-one-device-out diagnostic (train on one phone, validate on each), 3 repeats
    "configs/exp/L_D1_M2M.yaml",
    "configs/exp/L_D1_M2S.yaml",
    "configs/exp/L_D1_S2M.yaml",
    "configs/exp/L_D1_S2S.yaml",
    "configs/exp/L_D2_M2M.yaml",
    "configs/exp/L_D2_M2S.yaml",
    "configs/exp/L_D2_S2M.yaml",
    "configs/exp/L_D2_S2S.yaml",
    "configs/exp/L_D3_M2M.yaml",
    "configs/exp/L_D3_M2S.yaml",
    "configs/exp/L_D3_S2M.yaml",
    "configs/exp/L_D3_S2S.yaml",
    "configs/exp/L_D4_M2M.yaml",
    "configs/exp/L_D4_M2S.yaml",
    "configs/exp/L_D4_S2M.yaml",
    "configs/exp/L_D4_S2S.yaml",
    "configs/exp/L_O7_M2M.yaml",
    "configs/exp/L_O7_M2S.yaml",
    "configs/exp/L_O7_S2M.yaml",
    "configs/exp/L_O7_S2S.yaml",
]
DRIVE = "/content/drive/MyDrive/soilgsd"
REPO = "/content/repo"

In [ ]:
import os
from google.colab import drive, userdata

drive.mount("/content/drive")
os.makedirs(DRIVE, exist_ok=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv
!rm -rf {REPO} && git clone -q --branch {BRANCH} https://github.com/lev-laptinov/lab1-cv-acv-soil-grain-size.git {REPO}
%cd {REPO}
!git log -1 --format="%h %s"
# Colab already has torch, numpy, pandas, sklearn, scipy, pillow, matplotlib; soilgsd is imported from src/
!pip install -q -U timm==1.0.30 tabulate kaggle huggingface_hub
os.environ["PYTHONPATH"] = f"{REPO}/src"

In [ ]:
# Competition data (not committed to git) and the tile cache from an earlier session, if any
import shutil
from pathlib import Path

# New-style KGAT_ tokens go into KAGGLE_API_TOKEN (needs a recent kaggle CLI, upgraded above)
os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN")
!kaggle --version
!kaggle competitions download soil-grain-size-from-photos -p data/raw -q
!cd data/raw && unzip -q -o soil-grain-size-from-photos.zip && rm soil-grain-size-from-photos.zip
import numpy as np

# Tile cache from an earlier session: only files that open completely (a half-synced Drive copy is dropped)
os.makedirs("data/cache/tiles", exist_ok=True)
for f in sorted(Path(f"{DRIVE}/tiles").glob("*.npz")) if Path(f"{DRIVE}/tiles").exists() else []:
    try:
        with np.load(f) as z:
            z["tiles"][-1], z["sample_id"][-1]
        shutil.copy(f, f"data/cache/tiles/{f.name}")
        print("tile cache from Drive:", f.name)
    except Exception as e:
        print(f"skipping broken {f.name} on Drive ({e!r}); the tiles will be cut again")
        f.unlink()
# Earlier results kept on Drive (e.g. block B, not in git yet): new rows are added to them, not to the cloned table
!test -f {DRIVE}/results.csv && cp {DRIVE}/results.csv reports/results.csv; wc -l reports/results.csv

In [ ]:
# Warm-up before the parallel runs: cuts and caches the tiles once (parallel runs would all cut them at the
# same time) and shows speed / memory of one run on this GPU (no scores)
!python scripts/run_finetune.py {CONFIGS[0]} --smoke 20
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv

In [ ]:
import re
import shutil
import subprocess
import time
from concurrent.futures import FIRST_COMPLETED, ThreadPoolExecutor, wait
from pathlib import Path

import yaml

PARALLEL = 4  # runs on the GPU at once
os.makedirs("logs", exist_ok=True)
env = {**os.environ, "SOILGSD_PARALLEL": str(PARALLEL)}  # noted in the compute column of results.csv


def run(cfg: str) -> tuple[str, int]:
    name = cfg.split("/")[-1].split(".yaml")[0]
    with open(f"logs/{name}.log", "w") as log:
        done = subprocess.run(["python", "scripts/run_finetune.py", *cfg.split()], stdout=log,
                              stderr=subprocess.STDOUT, env=env)
    return name, done.returncode


def sync_to_drive():
    shutil.copytree("outputs", f"{DRIVE}/outputs", dirs_exist_ok=True)
    shutil.copytree("logs", f"{DRIVE}/logs", dirs_exist_ok=True)
    shutil.copy("reports/results.csv", f"{DRIVE}/results.csv")
    # Tiles: each file once, under a temporary name first, so Drive never holds a half-written cache
    os.makedirs(f"{DRIVE}/tiles", exist_ok=True)
    for f in Path("data/cache/tiles").glob("*.npz"):
        dst = Path(f"{DRIVE}/tiles/{f.name}")
        if ".tmp." not in f.name and (not dst.exists() or dst.stat().st_size != f.stat().st_size):
            shutil.copy(f, f"{dst}.part")
            os.replace(f"{dst}.part", dst)


def folds_planned(cfg: str) -> int:
    """Training folds of one run: 6 per CV repeat (--repeats on the line, else the config's repeats)."""
    path, *args = cfg.split()
    reps = [a for a in args[args.index("--repeats") + 1:]] if "--repeats" in args else yaml.safe_load(open(path)).get("repeats", [0, 1, 2])
    return 6 * len(reps)


def folds_done() -> int:
    """Finished folds over all runs, from the "repeat r fold k: ...s" lines of run_finetune.py."""
    return sum(len(re.findall(r"^repeat \d+ fold \d+:", Path(f).read_text(), re.M)) for f in Path("logs").glob("*.log"))


def status(start: float, total: int, runs_done: int) -> str:
    done, minutes = folds_done(), (time.time() - start) / 60
    eta = f"~{minutes / done * (total - done):.0f} min left" if done else "ETA after the first fold"
    return f"{minutes:5.0f} min | runs {runs_done}/{len(CONFIGS)} | folds {done}/{total} | {eta}"


total = sum(folds_planned(c) for c in CONFIGS)
start, runs_done = time.time(), 0
with ThreadPoolExecutor(PARALLEL) as pool:
    pending = {pool.submit(run, c) for c in CONFIGS}
    print(f"started: {len(CONFIGS)} runs, {total} folds, {PARALLEL} at a time; per-run logs in logs/<name>.log", flush=True)
    while pending:
        # Wake up when a run ends, or every minute for a progress line
        finished, pending = wait(pending, timeout=60, return_when=FIRST_COMPLETED)
        for future in finished:
            runs_done += 1
            name, code = future.result()
            print(f"  {name}: {'ok' if code == 0 else f'FAILED (exit {code}), see logs/{name}.log'}", flush=True)
            sync_to_drive()  # here, in the main thread: one copy at a time
        print(status(start, total, runs_done), flush=True)
print("done; results in", DRIVE)

In [ ]:
import pandas as pd

names = [c.split("/")[-1].split(".yaml")[0] for c in CONFIGS]
res = pd.read_csv("reports/results.csv").set_index("name")
cols = ["backbone", "emd", "emd_std_folds", "emd_fine", "emd_sand", "emd_gravel", "D50_log_err", "repeats", "compute"]
res.loc[res.index.intersection(names), cols].sort_values("emd")